# Linear Convection: FTBS (CFD) vs PINN

Solve  $u_t + c\,u_x = 0$  two ways and compare, with **separate timing** for each:

- **(A) FTBS** finite difference (classic CFD)
- **(B) PINN** in PyTorch, **no labeled data** (physics + IC/BC only)

Initial condition: a smooth **Gaussian** pulse (fast to train). Runs in a few seconds on Colab.

> Tip: Runtime -> Change runtime type -> **GPU** (optional; it also runs fine on CPU).

In [ ]:
# Cell 1 -- Imports, device, problem definition
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

c, L, T = 1.0, 2.0, 1.0        # speed, domain [0,L], final time

def u0(x):                     # smooth Gaussian -> easy for a small PINN
    exp = torch.exp if torch.is_tensor(x) else np.exp
    return exp(-((x - 0.5) ** 2) / (2 * 0.1 ** 2))

def exact(x, t):
    return u0(x - c * t)

In [ ]:
# Cell 2 -- PART A: FTBS (CFD)   u_i^{n+1} = u_i^n - CFL*(u_i^n - u_{i-1}^n)
nx = 201; dx = L / (nx - 1); x = np.linspace(0, L, nx)
CFL = 0.8; dt = CFL * dx / c; nt = int(round(T / dt))

t0 = time.perf_counter()
u = u0(x).copy()
for n in range(nt):
    u[1:] = u[1:] - CFL * (u[1:] - u[:-1])
    u[0]  = 0.0                # inflow: nothing enters at x=0
cfd_time = time.perf_counter() - t0

u_ftbs = u; t_final = nt * dt
print(f'FTBS (CFD) time: {cfd_time:.4f} s   (nt={nt} steps)')

In [ ]:
# Cell 3 -- PART B: small PINN (no labeled data)
class PINN(nn.Module):
    def __init__(self, h=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, h), nn.Tanh(),
                                 nn.Linear(h, h), nn.Tanh(),
                                 nn.Linear(h, 1))
    def forward(self, x, t):
        return self.net(torch.cat([x, t], 1))

model = PINN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
mse = nn.MSELoss()

# Fixed IC/BC points (cheap); resample interior each step
x_ic = torch.rand(500, 1, device=device) * L
t_ic = torch.zeros_like(x_ic);  u_ic = u0(x_ic)
t_bc = torch.rand(500, 1, device=device) * T
x_bc = torch.zeros_like(t_bc);  u_bc = exact(x_bc, t_bc)

t0 = time.perf_counter()
for e in range(1500):
    opt.zero_grad()
    xi = (torch.rand(2000, 1, device=device) * L).requires_grad_(True)
    ti = (torch.rand(2000, 1, device=device) * T).requires_grad_(True)
    ui = model(xi, ti)
    u_t = torch.autograd.grad(ui, ti, torch.ones_like(ui), create_graph=True)[0]
    u_x = torch.autograd.grad(ui, xi, torch.ones_like(ui), create_graph=True)[0]
    loss = mse(u_t + c * u_x, torch.zeros_like(ui)) \
         + 10 * mse(model(x_ic, t_ic), u_ic) \
         + 10 * mse(model(x_bc, t_bc), u_bc)
    loss.backward(); opt.step()
    if e % 300 == 0:
        print(f'epoch {e:4d}  loss {loss.item():.2e}')
if device.type == 'cuda':
    torch.cuda.synchronize()   # make sure GPU work is done before timing
pinn_time = time.perf_counter() - t0
print(f'\nPINN training time: {pinn_time:.4f} s')

with torch.no_grad():
    xe = torch.tensor(x, dtype=torch.float32, device=device).reshape(-1, 1)
    u_pinn = model(xe, torch.full_like(xe, t_final)).cpu().numpy().ravel()

In [ ]:
# Cell 4 -- PART C: compare accuracy and timing
u_ex = exact(x, t_final)
l2_ftbs = np.sqrt(np.mean((u_ftbs - u_ex) ** 2))
l2_pinn = np.sqrt(np.mean((u_pinn - u_ex) ** 2))

print('================  SUMMARY  ================')
print(f'FTBS (CFD) : time = {cfd_time:8.4f} s   L2 error = {l2_ftbs:.3e}')
print(f'PINN       : time = {pinn_time:8.4f} s   L2 error = {l2_pinn:.3e}')
print(f'PINN / CFD time ratio = {pinn_time / cfd_time:.1f}x')

plt.figure(figsize=(10, 4.5))
plt.plot(x, u0(x), 'k:', label='initial')
plt.plot(x, u_ex, 'g', lw=2.5, label='exact')
plt.plot(x, u_ftbs, 'b--', label=f'FTBS ({cfd_time:.3f}s)')
plt.plot(x, u_pinn, 'r-.', label=f'PINN ({pinn_time:.2f}s)')
plt.xlabel('x'); plt.ylabel('u'); plt.legend(); plt.grid(alpha=.3)
plt.title('Linear convection: FTBS (CFD) vs PINN')
plt.tight_layout(); plt.show()